<a href="https://colab.research.google.com/github/sflores14/inspirastem2026-bioquimica-computacional/blob/main/day3/03_dinamica_molecular.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/sflores14/inspirastem2026-bioquimica-computacional/blob/main/day3/03_dinamica_molecular.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Día 3 | De una pose estática a una trayectoria molecular

**InspiraSTEM 2026 | Bioquímica Computacional Aplicada**

### Pregunta del día

> **Una pose predicha sigue siendo plausible cuando permitimos que el sistema se mueva?**

En los dos días anteriores construimos una hipótesis:

$$
\text{estructura}
\rightarrow
\text{pocket}
\rightarrow
\text{química}
\rightarrow
\text{Boltz-2}
\rightarrow
\text{GNINA}
\rightarrow
\text{interacciones}
$$

Hoy agregamos una dimensión que faltaba:

$$
\boxed{\text{tiempo}}
$$

La dinámica molecular no produce una sola estructura. Produce una **trayectoria**, una colección ordenada de configuraciones moleculares.

### Ruta de hoy

| Tiempo aproximado | Actividad |
|---|---|
| 0 a 7 min | Un ejemplo biomédico real: KRAS G12C y sotorasib |
| 7 a 30 min | Fuerzas, energía, timestep y temperatura con modelos simples |
| 30 a 45 min | Cómo se construye una simulación biomolecular |
| 45 a 65 min | Elegir condiciones y correr MD corta de T4 L99A |
| 65 a 90 min | Visualizar y analizar la trayectoria |
| 90 a 103 min | Comparar los tres ligandos y revelar el experimento |
| 103 a 113 min | Capstone: dinámica de las flaps de HIV-1 protease |
| 113 a 120 min | Elegir un target de enfermedad y comenzar una MD propia |

**Regla del día:** una trayectoria corta aporta evidencia sobre comportamiento estructural dentro del tiempo que muestreamos. No convierte una predicción en una medición experimental de afinidad.


In [ ]:
#@title Revisar el runtime
import os, sys, subprocess, platform, shutil
from pathlib import Path

print("Python:", platform.python_version())
print("Espacio disponible:")
subprocess.run(["df", "-h", "/content"], check=False)

try:
    gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True).strip()
    print("GPU:", gpu)
except Exception:
    print("GPU: no detectada. El análisis y los modelos simples pueden correr en CPU.")


In [ ]:
#@title Preparar herramientas para el Día 3
#@markdown Instala las herramientas necesarias para simulación, visualización y análisis.
import os, sys, subprocess, shutil, importlib.util, json, urllib.request
from pathlib import Path

ROOT=Path('/content/inspirastem_day3')
ROOT.mkdir(parents=True,exist_ok=True)

slug_map={'Benzene':'benzene','Toluene':'toluene','n-Propylbenzene':'n_propylbenzene'}
display_name={'benzene':'Benzene','toluene':'Toluene','n_propylbenzene':'n-Propylbenzene'}
REFERENCE_URL_DEFAULT = (
    'https://github.com/sflores14/inspirastem2026-bioquimica-computacional/'
    'releases/download/day3-data-v1/day3_reference_results.zip'
)

packages = [
    ('openmm','openmm'), ('MDAnalysis','MDAnalysis'),
    ('py3Dmol','py3Dmol'), ('rdkit','rdkit'),
    ('pandas','pandas'), ('matplotlib','matplotlib'),
    ('scipy','scipy'), ('requests','requests'), ('tqdm','tqdm')
]
for pkg,imp in packages:
    if importlib.util.find_spec(imp) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])

# AmberTools vive en un entorno Pixi separado para evitar reiniciar Colab.
PIXI=Path.home()/'.pixi/bin/pixi'
if not PIXI.exists():
    subprocess.check_call('curl -fsSL https://pixi.sh/install.sh | bash',shell=True)
AMBER_ENV=ROOT/'ambertools_env'
if not (AMBER_ENV/'pixi.toml').exists():
    subprocess.check_call([str(PIXI),'init',str(AMBER_ENV),'-c','conda-forge'])
subprocess.check_call([str(PIXI),'add','--manifest-path',str(AMBER_ENV/'pixi.toml'),'ambertools'],stdout=subprocess.DEVNULL)

def amber(args, cwd=None, capture=False):
    cmd=[str(PIXI),'run','--manifest-path',str(AMBER_ENV/'pixi.toml')]+list(args)
    return subprocess.run(cmd,cwd=cwd,text=True,check=True,
                          stdout=subprocess.PIPE if capture else None,
                          stderr=subprocess.STDOUT if capture else None)

import openmm as mm
print('OpenMM:',mm.version.version)
print('Plataformas:',[mm.Platform.getPlatform(i).getName() for i in range(mm.Platform.getNumPlatforms())])
print('AmberTools:',amber(['which','tleap'],capture=True).stdout.strip())
print('Herramientas listas.')


# Antes de empezar | Por qué nos importa el movimiento molecular

Vamos a comenzar con un sistema biomédico real.

**KRAS** es una proteína humana de señalización. La mutación **G12C** aparece en varios cánceres y crea una cisteína que puede ser aprovechada por inhibidores covalentes. **AMG 510**, posteriormente llamado **sotorasib**, se une a KRAS G12C y se usa clínicamente en cánceres con esta alteración.

Un estudio publicado de dinámica molecular ejecutó múltiples simulaciones de **5 a 10 microsegundos** de KRAS G12C con AMG 510 y depositó públicamente las trayectorias, películas y estructuras representativas.

Aquí no vamos a descargar decenas de gigabytes. Usaremos cuatro **estados metastables extraídos de esas trayectorias reales**.

Importante:

> Los cuatro estados siguientes no son cuatro frames consecutivos. Son conformaciones representativas identificadas al analizar trayectorias largas de MD.

Mientras los miras, piensa:

> Si una estructura cristalográfica es una fotografía, qué información adicional obtenemos cuando una proteína puede visitar varias conformaciones?

Fuente de los datos: Pantsar T. *KRAS(G12C)-AMG 510 Interaction Dynamics Revealed by All-Atom Molecular Dynamics Simulations.* Scientific Reports 2020. Datos públicos: https://doi.org/10.5281/zenodo.3711537


In [ ]:
#@title Ver estados de KRAS G12C + AMG 510 derivados de MD

import requests, py3Dmol
from pathlib import Path

KRAS_DIR = ROOT / 'kras_example'
KRAS_DIR.mkdir(parents=True, exist_ok=True)

state_urls = {
    'S1': 'https://zenodo.org/records/3711537/files/S1_01.pdb?download=1',
    'S2': 'https://zenodo.org/records/3711537/files/S2_01.pdb?download=1',
    'S3': 'https://zenodo.org/records/3711537/files/S3_01.pdb?download=1',
    'S4': 'https://zenodo.org/records/3711537/files/S4_01.pdb?download=1',
}

kras_state = "S2" #@param ["S1", "S2", "S3", "S4"]
show_surface = True #@param {type:"boolean"}
surface_opacity = 0.47 #@param {type:"slider", min:0.05, max:0.50, step:0.01}

for label, url in state_urls.items():
    path = KRAS_DIR / f'{label}.pdb'
    if not path.exists():
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        path.write_bytes(r.content)

pdbtxt = (KRAS_DIR / f'{kras_state}.pdb').read_text()

view = py3Dmol.view(width=900, height=560)
view.setBackgroundColor('white')
view.addModel(pdbtxt, 'pdb')

# Make protein cartoon style more defined
view.setStyle(
    {'protein': True},
    {'cartoon': {'color': 'gray', 'opacity': 0.95, 'style': 'oval'}}
)

# Make surface clear and easier to see with adjusted defaults
if show_surface:
    view.addSurface(
        py3Dmol.VDW,
        {'opacity': float(surface_opacity), 'color': 'skyblue'},
        {'protein': True}
    )

# AMG 510 in the corresponding crystal structure uses component ID MOV.
view.setStyle(
    {'resn': 'MOV'},
    {
        'stick': {'colorscheme': 'orangeCarbon', 'radius': 0.28},
        'sphere': {'colorscheme': 'orangeCarbon', 'scale': 0.14}
    }
)

# GDP and Mg are biologically relevant parts of the KRAS nucleotide state.
view.setStyle(
    {'resn': 'GDP'},
    {'stick': {'colorscheme': 'greenCarbon', 'radius': 0.18}}
)
view.setStyle(
    {'resn': 'MG'},
    {'sphere': {'color': 'magenta', 'radius': 0.7}}
)

# Highlight the G12C site.
view.setStyle(
    {'resi': 12},
    {
        'stick': {'colorscheme': 'yellowCarbon', 'radius': 0.22}
    }
)
view.addLabel(
    'Cys12',
    {
        'selection': {'resi': 12, 'atom': 'CA'},
        'backgroundColor': 'white',
        'fontColor': 'black',
        'fontSize': 11
    }
)

# Focus on the ligand if present, otherwise show the protein.
view.zoomTo({'resn': 'MOV'})
view.show()

print('Estado representativo:', kras_state)
print('S1-S4 son estados metastables derivados de simulaciones largas, no frames consecutivos.')

### Observación rápida

Cambia entre S1, S2, S3 y S4.

No necesitas interpretar cada movimiento todavía. Solo identifica la idea principal:

$$
\text{una proteína}
\neq
\text{una estructura rígida}
$$

Una proteína puede explorar un **ensemble** de conformaciones.

Ahora vamos a construir la intuición física necesaria para entender de dónde sale ese movimiento.


# Parte I | Antes de tocar una proteína: qué hace realmente MD

En dinámica molecular clásica tratamos a los átomos como partículas que sienten fuerzas.

La idea central se puede escribir con la segunda ley de Newton:

$$
m_i\frac{d^2\mathbf r_i}{dt^2} = \mathbf F_i
$$

La fuerza sobre cada átomo viene de cómo cambia la energía potencial del sistema:

$$
\mathbf F_i = -\nabla_i U(\mathbf r_1,\mathbf r_2,\ldots,\mathbf r_N)
$$

Así que una simulación necesita tres cosas:

1. una función de energía $U$;
2. una forma de convertir esa energía en fuerzas;
3. un integrador que avance posiciones y velocidades en pasos pequeños de tiempo.

Los tres modelos siguientes separan esas ideas antes de juntarlas en una proteína real.


## Modelo 1 | Un enlace como un resorte

Empecemos con una sola partícula unida a una posición de equilibrio.

Usaremos un potencial armónico:

$$
U(x)
=
\frac{1}{2}k(x-x_{eq})^2
$$

donde:

- $x$ es la posición actual;
- $x_{eq}$ es la posición de equilibrio;
- $k$ controla qué tan rígido es el "resorte".

La fuerza se obtiene de la pendiente de la energía:

$$
F(x)
=
-\frac{dU}{dx}
=
-k(x-x_{eq})
$$

Si alejamos la partícula del equilibrio, la fuerza apunta de regreso hacia $x_{eq}$.

Para este modelo también podemos anticipar la frecuencia natural:

$$
\omega = \sqrt{\frac{k}{m}}
$$

y el período aproximado:

$$
T = 2\pi\sqrt{\frac{m}{k}}
$$

Así que antes de ejecutar ya podemos predecir que aumentar $k$ acelera la oscilación y aumentar la masa $m$ la hace más lenta.

### Cómo avanzamos el tiempo

El código usa una forma de **Velocity Verlet**. De manera simplificada:

$$
v\left(t+\frac{\Delta t}{2}\right)
=
v(t)+\frac{F(t)}{2m}\Delta t
$$

$$
x(t+\Delta t)
=
x(t)
+
v\left(t+\frac{\Delta t}{2}\right)\Delta t
$$

Luego recalculamos la fuerza y completamos la actualización de la velocidad.

Prueba diferentes valores de `k`, `mass` y `dt`. El objetivo es ver físicamente qué significa integrar una ecuación de movimiento.


In [ ]:
#@title Modelo 1: partícula en un potencial armónico
#@markdown Cambia un parámetro, vuelve a ejecutar y compara el movimiento.
k = 175 #@param {type:"slider", min:5, max:200, step:5}
mass = 3.5 #@param {type:"slider", min:0.5, max:5.0, step:0.5}
x0 = 1.5 #@param {type:"slider", min:0.2, max:3.0, step:0.1}
equilibrium = 0.5 #@param {type:"slider", min:-1.0, max:1.0, step:0.1}
dt = 0.005 #@param [0.001, 0.002, 0.005, 0.01, 0.02]
steps = 800 #@param {type:"slider", min:200, max:1600, step:100}

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

x=float(x0); v=0.0
xs=[]; vs=[]; pe=[]; ke=[]
for _ in range(int(steps)):
    f=-float(k)*(x-float(equilibrium))
    v += 0.5*float(dt)*f/float(mass)
    x += float(dt)*v
    f2=-float(k)*(x-float(equilibrium))
    v += 0.5*float(dt)*f2/float(mass)
    xs.append(x); vs.append(v)
    pe.append(0.5*float(k)*(x-float(equilibrium))**2)
    ke.append(0.5*float(mass)*v*v)

fig, ax=plt.subplots(figsize=(8,3.4))
ax.plot(pe, label='Energía potencial')
ax.plot(ke, label='Energía cinética')
ax.plot(np.array(pe)+np.array(ke), label='Energía total', linewidth=2)
ax.set_xlabel('Paso')
ax.set_ylabel('Energía, unidades arbitrarias')
ax.legend()
plt.show()

fig2, ax2=plt.subplots(figsize=(7,2.4))
ax2.set_xlim(-3.5,3.5); ax2.set_ylim(-1,1); ax2.set_yticks([])
ax2.axvline(float(equilibrium), linestyle='--', linewidth=1)
pt,=ax2.plot([],[], 'o', markersize=14)
frames=np.linspace(0,len(xs)-1,min(120,len(xs))).astype(int)
def upd(i):
    pt.set_data([xs[frames[i]]],[0])
    return (pt,)
ani=FuncAnimation(fig2, upd, frames=len(frames), interval=35, blit=True)
plt.close(fig2)
display(HTML(ani.to_jshtml()))


### Prueba rápida

Cambia un parámetro a la vez.

1. Aumenta `k`.
2. Vuelve al valor original y aumenta `mass`.
3. Finalmente prueba un `dt` grande.

Antes de ejecutar, predice:

- cuál cambio hará que la partícula oscile más rápido;
- cuál hará que oscile más lento;
- qué puede pasar con la energía total si el paso de integración es demasiado grande.

La idea importante no es memorizar un número. Es ver que el movimiento calculado depende tanto del modelo físico como de la forma numérica en que avanzamos el tiempo.


## Modelo 2 | Interacciones sin un enlace: Lennard-Jones

Ahora quitamos el resorte. Las partículas no están unidas covalentemente.

Usaremos el potencial de Lennard-Jones:

$$
U(r)
=
4\epsilon
\left[
\left(\frac{\sigma}{r}\right)^{12}
-
\left(\frac{\sigma}{r}\right)^6
\right]
$$

donde:

- $r$ es la distancia entre las partículas;
- $\epsilon$ controla la profundidad del mínimo de energía;
- $\sigma$ controla la escala de distancia.

La fuerza vuelve a ser el negativo de la derivada:

$$
F(r)
=
-\frac{dU}{dr}
=
\frac{24\epsilon}{r}
\left[
2\left(\frac{\sigma}{r}\right)^{12}
-
\left(\frac{\sigma}{r}\right)^6
\right]
$$

El mínimo del potencial aparece en:

$$
r_{\min}=2^{1/6}\sigma
$$

Esto produce tres regiones fáciles de reconocer:

$$
\text{muy cerca: repulsión}
\quad\rightarrow\quad
\text{distancia intermedia: atracción}
\quad\rightarrow\quad
\text{muy lejos: interacción pequeña}
$$

### Antes de ejecutar

Prueba mover `r_start`.

- Qué esperas si empiezas muy cerca?
- Qué esperas cerca de $r_{\min}$?
- Qué cambia si aumentas $\epsilon$?

Este tipo de potencial 12-6 también aparece en force fields biomoleculares para describir parte de las interacciones no enlazadas.


In [ ]:
#@title Modelo 2: dos partículas con Lennard-Jones
#@markdown `epsilon` controla la profundidad del mínimo. `sigma` controla la escala de distancia.
epsilon = 1.9 #@param {type:"slider", min:0.2, max:3.0, step:0.1}
sigma = 0.8 #@param {type:"slider", min:0.5, max:2.0, step:0.1}
r_start = 2.0 #@param {type:"slider", min:0.75, max:4.0, step:0.05}
v_start = -0.15 #@param {type:"slider", min:-1.0, max:1.0, step:0.05}
dt_lj = 0.002 #@param [0.0005, 0.001, 0.002, 0.005]
steps_lj = 1500 #@param {type:"slider", min:500, max:3000, step:250}

import numpy as np, matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

eps=float(epsilon); sig=float(sigma)
def U(r):
    r=max(r,0.25)
    return 4*eps*((sig/r)**12-(sig/r)**6)
def Fmag(r):
    r=max(r,0.25)
    return 24*eps*(2*(sig**12)/(r**13)-(sig**6)/(r**7))

r=float(r_start); v=float(v_start)
rs=[]; us=[]; kes=[]
for _ in range(int(steps_lj)):
    f=Fmag(r)
    v += 0.5*float(dt_lj)*f
    r += float(dt_lj)*v
    r=max(r,0.45*sig)
    f2=Fmag(r)
    v += 0.5*float(dt_lj)*f2
    rs.append(r); us.append(U(r)); kes.append(0.5*v*v)

rgrid=np.linspace(0.75*sig,4*sig,500)
fig, ax=plt.subplots(figsize=(7.5,3.5))
ax.plot(rgrid,[U(x) for x in rgrid])
ax.axvline(2**(1/6)*sig, linestyle='--', linewidth=1, label='mínimo')
ax.axvline(r_start, linestyle=':', linewidth=1, label='distancia inicial')
ax.set_ylim(-1.5*eps,5*eps)
ax.set_xlabel('Distancia')
ax.set_ylabel('Energía potencial')
ax.legend()
plt.show()

fig2, ax2=plt.subplots(figsize=(7,2.4))
ax2.set_xlim(-0.5, max(4.5, max(rs)+0.5)); ax2.set_ylim(-1,1); ax2.set_yticks([])
p1,=ax2.plot([0],[0],'o',markersize=14)
p2,=ax2.plot([],[],'o',markersize=14)
frames=np.linspace(0,len(rs)-1,min(140,len(rs))).astype(int)
def upd(i):
    p2.set_data([rs[frames[i]]],[0]); return p1,p2
ani=FuncAnimation(fig2,upd,frames=len(frames),interval=30,blit=True)
plt.close(fig2)
display(HTML(ani.to_jshtml()))


### Conexión con nuestros ligandos

Benzene, toluene y n-propylbenzene son moléculas hidrofóbicas. En la cavidad L99A, el empaquetamiento y las interacciones no covalentes son una parte importante de la historia.

El modelo de Lennard-Jones no representa toda la química de la cavidad. En un force field real también aparecen cargas, enlaces, ángulos y torsiones.

Lo que sí nos deja ver es algo útil:

- a distancias muy cortas aparece una repulsión fuerte;
- a una distancia intermedia existe una región favorable;
- a distancias grandes la interacción se vuelve pequeña.

Una pose de docking puede verse bien en una imagen. MD nos deja preguntar si esa geometría sigue siendo razonable cuando todos los átomos empiezan a sentir fuerzas.


## Modelo 3 | Temperatura y energía cinética

En un sistema clásico, la energía cinética es:

$$
K
=
\sum_i
\frac{1}{2}m_i v_i^2
$$

La temperatura está relacionada con la energía cinética promedio. Para un sistema clásico idealizado:

$$
\langle K\rangle
=
\frac{f}{2}k_B T
$$

donde $f$ es el número de grados de libertad y $k_B$ es la constante de Boltzmann.

Por eso una escala típica de velocidad cambia aproximadamente como:

$$
v_{\mathrm{típica}}
\propto
\sqrt{\frac{T}{m}}
$$

El control `temperature_scale` de este juguete **no está calibrado en Kelvin**. Solo aumenta o disminuye la escala de las velocidades para que podamos ver la relación entre movimiento y energía cinética.

### Antes de ejecutar

Compara un valor bajo y uno alto de `temperature_scale`.

- Las partículas recorren más distancia entre frames?
- Aumenta linealmente la velocidad cuando duplicas la escala?
- Qué crees que tendría que hacer un thermostat en una simulación real?

Después llevaremos esta idea a OpenMM usando una temperatura real en Kelvin.


In [ ]:
#@title Modelo 3: temperatura y movimiento
#@markdown Este modelo usa partículas simples en 2D para visualizar el efecto de la energía cinética.
particle_count = 24 #@param {type:"slider", min:8, max:50, step:2}
temperature_scale = 1.9 #@param {type:"slider", min:0.2, max:3.0, step:0.1}
box_size_toy = 10.0 #@param {type:"slider", min:6, max:16, step:1}
frames_toy = 100 #@param {type:"slider", min:40, max:160, step:20}

import numpy as np, matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
rng=np.random.default_rng(14)
N=int(particle_count); L=float(box_size_toy)
pos=rng.uniform(0.5,L-0.5,(N,2))
vel=rng.normal(0,float(temperature_scale),(N,2))*0.07
history=[]
for f in range(int(frames_toy)):
    pos += vel
    for d in range(2):
        low=pos[:,d]<0.2; high=pos[:,d]>L-0.2
        vel[low|high,d]*=-1
        pos[:,d]=np.clip(pos[:,d],0.2,L-0.2)
    history.append(pos.copy())

fig,ax=plt.subplots(figsize=(4.8,4.8)); ax.set_xlim(0,L); ax.set_ylim(0,L)
sc=ax.scatter([],[],s=35); ax.set_xlabel('x'); ax.set_ylabel('y')
def upd(i):
    sc.set_offsets(history[i]); return (sc,)
ani=FuncAnimation(fig,upd,frames=len(history),interval=40,blit=True)
plt.close(fig)
display(HTML(ani.to_jshtml()))
print('Velocidad RMS:', round(float(np.sqrt(np.mean(vel**2))),4))


## Juntando las piezas | De los juguetes a una proteína

Un force field biomolecular combina muchos términos de energía al mismo tiempo:

$$
U_{\mathrm{total}}
=
U_{\mathrm{bonds}}
+
U_{\mathrm{angles}}
+
U_{\mathrm{dihedrals}}
+
U_{\mathrm{vdW}}
+
U_{\mathrm{electrostatics}}
$$

Por ejemplo:

$$
U_{\mathrm{bond}}
=
\sum_{\mathrm{bonds}}
\frac{1}{2}k_b(r-r_0)^2
$$

$$
U_{\mathrm{angle}}
=
\sum_{\mathrm{angles}}
\frac{1}{2}k_\theta(\theta-\theta_0)^2
$$

$$
U_{\mathrm{dihedral}}
=
\sum
\frac{V_n}{2}
\left[
1+\cos(n\phi-\delta)
\right]
$$

y para pares de átomos no enlazados aparece una combinación del término de Lennard-Jones y electrostática:

$$
U_{ij}^{\mathrm{nonbonded}}
=
4\epsilon_{ij}
\left[
\left(\frac{\sigma_{ij}}{r_{ij}}\right)^{12}
-
\left(\frac{\sigma_{ij}}{r_{ij}}\right)^6
\right]
+
\frac{1}{4\pi\epsilon_0}
\frac{q_iq_j}{r_{ij}}
$$

OpenMM repite esencialmente este ciclo:

$$
\boxed{
\text{coordenadas}
\rightarrow
U
\rightarrow
\mathbf F
\rightarrow
\text{integración}
\rightarrow
\text{nuevas coordenadas}
}
$$

Después de cierto número de pasos guardamos un **frame**. La colección ordenada de esos frames es la trayectoria.

Eso es lo que vamos a hacer ahora con T4 lysozyme L99A.


# Parte II | De los modelos simples a T4 lysozyme L99A

En los modelos anteriores controlamos una o dos interacciones. Una proteína contiene miles de átomos y un force field combina muchos términos a la vez.

Ahora volvemos a nuestro sistema de los tres días:

- T4 lysozyme L99A
- Benzene
- Toluene
- n-Propylbenzene

Las poses iniciales provienen del protocolo de GNINA del Día 2. No repetiremos docking durante la clase.

## Estructura, topología y trayectoria

Conviene separar tres ideas:

**Estructura:** una fotografía de coordenadas atómicas en un momento.

**Topología:** qué átomos existen, cómo se conectan, sus tipos y los parámetros necesarios para calcular energía y fuerzas.

**Trayectoria:** coordenadas de esos mismos átomos a través del tiempo:

$$
\mathbf R(t_0),\mathbf R(t_1),\mathbf R(t_2),\ldots,\mathbf R(t_n)
$$

En nuestro caso, el archivo de topología describe el sistema y los archivos XTC contienen las coordenadas guardadas durante la simulación.

Antes de calcular cualquier métrica, primero vamos a mirar una trayectoria.


## Trayectorias de referencia para toda la clase

Ya generamos previamente trayectorias de referencia para **Benzene, Toluene y n-Propylbenzene** usando el mismo protocolo.

El notebook las descarga automáticamente. Así todos pueden completar el análisis aunque un grupo no consiga GPU o su simulación corta falle.

La corrida en vivo de tu grupo es una demostración del proceso. Para comparar los tres ligandos usaremos las referencias precomputadas.


In [ ]:
#@title Cargar las trayectorias de referencia
import urllib.request, zipfile, shutil, json
from pathlib import Path

ROOT=Path('/content/inspirastem_day3')
ROOT.mkdir(parents=True,exist_ok=True)

REFERENCE_URL = REFERENCE_URL_DEFAULT

local_zip=ROOT/'day3_reference_results.zip'

if not local_zip.exists():
    print('Descargando las trayectorias de referencia del workshop...')
    urllib.request.urlretrieve(REFERENCE_URL, local_zip)

print('ZIP:', local_zip)
print('Tamaño:', round(local_zip.stat().st_size/1024**2,1), 'MiB')

LOAD=ROOT/'loaded_reference'
if LOAD.exists():
    shutil.rmtree(LOAD)

with zipfile.ZipFile(local_zip) as z:
    z.extractall(LOAD)

REF_LOADED=LOAD/'day3_reference_results'
if not REF_LOADED.exists():
    raise FileNotFoundError('El ZIP no contiene la carpeta day3_reference_results esperada.')

meta_file=REF_LOADED/'metadata.json'
if meta_file.exists():
    reference_metadata=json.loads(meta_file.read_text())
    print('Sistema:',reference_metadata.get('system'))
    print('Producción por ligando:',reference_metadata.get('reference_ns'),'ns')
    print('Temperatura de referencia:',reference_metadata.get('temperature_K'),'K')
    print('NaCl de referencia:',reference_metadata.get('salt_target_M'),'M')
else:
    reference_metadata={}

print('Resultados listos.')


In [ ]:
#@title Elegir un ligando para el grupo
selected_ligand = "Toluene" #@param ["Benzene", "Toluene", "n-Propylbenzene"]
student_slug=slug_map[selected_ligand]
print('Ligando elegido:',selected_ligand)


## Elegir condiciones que se parezcan a un entorno biológico

Antes de correr la simulación, hagan una búsqueda rápida.

Busquen tres valores:

1. **Temperatura corporal humana aproximada** en grados Celsius y conviértanla a Kelvin:

$$
T(K)=T(^\circ C)+273.15
$$

2. **Concentración aproximada de NaCl** usada para representar una solución fisiológica.

3. **Presión atmosférica aproximada** en bar.

No existe una única condición que represente todas las células. Para este ejercicio solo queremos valores razonables para una proteína en solución acuosa.

Como referencia, valores cercanos a:

$$
T \approx 310\ \mathrm{K}
$$

$$
[\mathrm{NaCl}] \approx 0.15\ \mathrm{M}
$$

$$
P \approx 1\ \mathrm{bar}
$$

son elecciones comunes para aproximar condiciones fisiológicas.

### Antes de ejecutar

> Qué valor eligió tu grupo para temperatura?

> Qué valor eligió para sal?

> Qué valor eligió para presión?

> Esperas más o menos movimiento molecular si aumentamos la temperatura?


In [ ]:
#@title Elegir condiciones del grupo
student_temperature_K = 291 #@param {type:"slider", min:285.0, max:325.0, step:1.0}
student_salt_M = 0.08 #@param {type:"slider", min:0.0, max:0.30, step:0.01}
student_pressure_bar = 0.92 #@param {type:"slider", min:0.8, max:1.2, step:0.01}
student_water_padding_A = "10" #@param [10.0, 12.0, 14.0]

print('Condiciones elegidas')
print('Temperatura:',student_temperature_K,'K')
print('NaCl aproximado:',student_salt_M,'M')
print('Presión:',student_pressure_bar,'bar')
print('Padding de agua:',student_water_padding_A,'A')
print()
print('Pregunta para el grupo: qué valor investigaron primero y por qué lo consideran razonable?')


In [ ]:
#@title Preparar el sistema del grupo con esas condiciones
#@markdown Esta celda reutiliza los parámetros del ligando del ZIP, pero vuelve a solvatar y colocar iones para que `student_salt_M` tenga un efecto real.
prepare_group_system = True #@param {type:"boolean"}

from pathlib import Path
import json, shutil

STUDENT_SYSTEM=Path('/content/day3_student_system')
STUDENT_SYSTEM.mkdir(parents=True,exist_ok=True)

if not prepare_group_system:
    print('Preparación personalizada omitida. Se usará el sistema de referencia para la demostración.')
    student_topology = REF_LOADED/'systems'/student_slug/'system.prmtop'
    student_coordinates = REF_LOADED/'systems'/student_slug/'system.inpcrd'
else:
    if 'amber' not in globals():
        raise RuntimeError('Primero ejecuta la celda Preparar herramientas para el Día 3.')

    SRC=REF_LOADED/'systems'/student_slug
    protein_clean=SRC/'protein_clean.pdb'
    mol2=SRC/'ligand.mol2'
    frcmod=SRC/'ligand.frcmod'
    for f in [protein_clean,mol2,frcmod]:
        if not f.exists():
            raise FileNotFoundError(f'Falta {f.name} en el ZIP de referencia.')

    neutral_pdb=STUDENT_SYSTEM/'neutral_solvated.pdb'
    leap1=STUDENT_SYSTEM/'leap_stage1.in'
    leap1.write_text(f"""source leaprc.protein.ff14SB
source leaprc.gaff2
source leaprc.water.tip3p
loadamberparams {frcmod}
LIG = loadmol2 {mol2}
PROT = loadpdb {protein_clean}
SYS = combine {{PROT LIG}}
solvateBox SYS TIP3PBOX {float(student_water_padding_A):.3f}
addIons SYS Na+ 0
addIons SYS Cl- 0
savepdb SYS {neutral_pdb}
quit
""")
    amber(['tleap','-f',str(leap1)],cwd=STUDENT_SYSTEM,capture=True)

    wat=set()
    for line in neutral_pdb.read_text().splitlines():
        if line.startswith(('ATOM','HETATM')) and line[17:20].strip() in ('WAT','HOH'):
            wat.add((line[21:22],line[22:26],line[26:27]))
    n_wat=len(wat)
    n_pairs=max(0,int(round(n_wat*float(student_salt_M)/55.5)))

    student_topology=STUDENT_SYSTEM/'system.prmtop'
    student_coordinates=STUDENT_SYSTEM/'system.inpcrd'
    student_pdb=STUDENT_SYSTEM/'system.pdb'
    leap2=STUDENT_SYSTEM/'leap_final.in'
    leap2.write_text(f"""source leaprc.protein.ff14SB
source leaprc.gaff2
source leaprc.water.tip3p
loadamberparams {frcmod}
LIG = loadmol2 {mol2}
PROT = loadpdb {protein_clean}
SYS = combine {{PROT LIG}}
solvateBox SYS TIP3PBOX {float(student_water_padding_A):.3f}
addIons SYS Na+ 0
addIons SYS Cl- 0
addIonsRand SYS Na+ {n_pairs} Cl- {n_pairs}
check SYS
saveamberparm SYS {student_topology} {student_coordinates}
savepdb SYS {student_pdb}
quit
""")
    result=amber(['tleap','-f',str(leap2)],cwd=STUDENT_SYSTEM,capture=True)
    (STUDENT_SYSTEM/'tleap.log').write_text(result.stdout or '')
    print('Sistema listo.')
    print('Aguas:',n_wat)
    print('Pares NaCl añadidos:',n_pairs)
    print('Objetivo aproximado:',student_salt_M,'M')


In [ ]:
#@title Correr una demostración corta con el ligando del grupo
#@markdown En GPU, el grupo minimiza, equilibra brevemente y genera una trayectoria corta. En CPU, continúa con las referencias.
run_live_demo = True #@param {type:"boolean"}
live_equil_ps = "20" #@param [20.0, 50.0, 100.0]
live_ps = "25" #@param [25.0, 50.0, 100.0, 250.0]
allow_cpu_live = False #@param {type:"boolean"}
live_save_ps = "0.5" #@param [0.5, 1.0, 2.0, 5.0]

if run_live_demo:
    import time, openmm as mm
    from openmm import unit
    from openmm.app import AmberPrmtopFile, AmberInpcrdFile, Simulation, PME, HBonds, XTCReporter, StateDataReporter, PDBFile

    if 'student_topology' not in globals() or 'student_coordinates' not in globals():
        student_topology = REF_LOADED/'systems'/student_slug/'system.prmtop'
        student_coordinates = REF_LOADED/'systems'/student_slug/'system.inpcrd'
        print('Usando el sistema de referencia. Ejecuta la celda anterior si quieres aplicar otra concentración de sal.')

    names=[mm.Platform.getPlatform(i).getName() for i in range(mm.Platform.getNumPlatforms())]
    if 'CUDA' in names:
        live_platform=mm.Platform.getPlatformByName('CUDA'); live_props={'Precision':'mixed'}
    elif 'OpenCL' in names:
        live_platform=mm.Platform.getPlatformByName('OpenCL'); live_props={'Precision':'mixed'}
    else:
        live_platform=mm.Platform.getPlatformByName('CPU'); live_props={}

    if live_platform.getName()=='CPU' and not allow_cpu_live:
        print('No se detectó GPU. Se omite la corrida en vivo y se usan las trayectorias de referencia.')
    else:
        prmtop=AmberPrmtopFile(str(student_topology))
        inp=AmberInpcrdFile(str(student_coordinates))
        system=prmtop.createSystem(nonbondedMethod=PME,nonbondedCutoff=1.0*unit.nanometer,
                                   constraints=HBonds,rigidWater=True,ewaldErrorTolerance=0.0005)
        system.addForce(mm.MonteCarloBarostat(float(student_pressure_bar)*unit.bar,float(student_temperature_K)*unit.kelvin,25))
        integ=mm.LangevinMiddleIntegrator(float(student_temperature_K)*unit.kelvin,1.0/unit.picosecond,2.0*unit.femtoseconds)
        sim=Simulation(prmtop.topology,system,integ,live_platform,live_props)
        sim.context.setPositions(inp.positions)
        if inp.boxVectors is not None: sim.context.setPeriodicBoxVectors(*inp.boxVectors)

        print('Minimizando...')
        sim.minimizeEnergy(maxIterations=3000)
        sim.context.setVelocitiesToTemperature(float(student_temperature_K)*unit.kelvin,2026)
        eq_steps=int(float(live_equil_ps)*1000/2.0)
        print('Equilibrando',live_equil_ps,'ps a',student_temperature_K,'K y',student_pressure_bar,'bar...')
        sim.step(eq_steps)

        live_dir=Path('/content/day3_live'); live_dir.mkdir(exist_ok=True)
        live_xtc=live_dir/f'{student_slug}_live.xtc'; live_log=live_dir/f'{student_slug}_live.csv'
        live_pdb=live_dir/f'{student_slug}_start.pdb'
        state=sim.context.getState(getPositions=True,enforcePeriodicBox=True)
        with open(live_pdb,'w') as f: PDBFile.writeFile(prmtop.topology,state.getPositions(),f,keepIds=True)

        report_steps=max(1,int(float(live_save_ps)*1000/2.0))
        steps=int(float(live_ps)*1000/2.0)
        sim.reporters=[XTCReporter(str(live_xtc),report_steps,enforcePeriodicBox=True),
                       StateDataReporter(str(live_log),report_steps,step=True,time=True,potentialEnergy=True,
                                         temperature=True,density=True,speed=True,totalSteps=steps,separator=',')]
        t0=time.time(); sim.step(steps); elapsed=time.time()-t0
        nsday=(float(live_ps)/1000)/(elapsed/86400)
        print('Plataforma:',live_platform.getName())
        print('Temperatura:',student_temperature_K,'K')
        print('Presión:',student_pressure_bar,'bar')
        print('NaCl objetivo:',student_salt_M,'M')
        print('Tiempo simulado:',live_ps,'ps')
        print('Rendimiento:',round(nsday,1),'ns/day')
        print('Trayectoria en vivo:',live_xtc)
else:
    print('Demo en vivo omitido. Continúa con la trayectoria de referencia.')


## Ver una trayectoria antes de convertirla en números

Primero miren el movimiento.

La proteína se mostrará como **cartoon** dentro de una **superficie transparente**. El ligando aparece como sticks y una pequeña selección de moléculas de agua cercanas se conserva para recordar que la proteína no está flotando en vacío.

El viewer alinea cada frame usando el backbone de la proteína. Sin esa alineación, la traslación y rotación globales pueden ocultar el movimiento interno que queremos estudiar.

La superficie es una ayuda visual. Las aguas que se muestran son solo una pequeña selección del solvente real.


In [ ]:
#@title Ver una trayectoria real de T4 L99A
#@markdown Construye una animación ligera a partir de la trayectoria completa del ligando elegido.
surface_opacity = 0.16 #@param {type:"slider", min:0.05, max:0.40, step:0.01}
cartoon_opacity = 0.90 #@param {type:"slider", min:0.3, max:1.0, step:0.05}
show_ligand = True #@param {type:"boolean"}
show_waters = True #@param {type:"boolean"}
nearby_waters = 45 #@param {type:"slider", min:10, max:100, step:5}
viewer_frames = 90 #@param [45, 60, 90, 120]

import numpy as np
import MDAnalysis as mda
from MDAnalysis.analysis import align
from MDAnalysis.lib.distances import distance_array
import py3Dmol
from pathlib import Path

D=REF_LOADED/'systems'/student_slug
R=REF_LOADED/'runs'/student_slug/'production'
trjs=[str(p) for p in sorted(R.glob('traj_*.xtc'))]
if not trjs:
    raise FileNotFoundError('No se encontraron trayectorias XTC para '+student_slug)

u=mda.Universe(str(D/'system.prmtop'),trjs)
ref=mda.Universe(str(D/'system.prmtop'),trjs[0])
ref.trajectory[0]
u.trajectory[0]

protein=u.select_atoms('protein')
ligand=u.select_atoms('resname LIG')
ligheavy=u.select_atoms('resname LIG and not name H*')

waterO=u.select_atoms(
    '(resname WAT or resname HOH or resname SOL) and '
    '(name O or name OW or name OH2)'
)

selected_water_resindices=[]
if show_waters and len(waterO) and len(ligheavy):
    dw=distance_array(waterO.positions,ligheavy.positions,box=u.dimensions)
    mind=dw.min(axis=1)
    order=np.argsort(mind)[:min(int(nearby_waters),len(waterO))]
    selected_water_resindices=sorted(set(int(x) for x in waterO[order].resindices))

water_atoms=(
    u.atoms[np.isin(u.atoms.resindices,selected_water_resindices)]
    if selected_water_resindices else u.atoms[[]]
)

sel=protein + ligand + water_atoms

cache=ROOT/f'viewer_{student_slug}_{int(viewer_frames)}f_{len(selected_water_resindices)}w.pdb'
frame_idx=np.unique(np.linspace(0,len(u.trajectory)-1,min(int(viewer_frames),len(u.trajectory))).astype(int))

with mda.Writer(str(cache),sel.n_atoms,multiframe=True) as W:
    for i in frame_idx:
        u.trajectory[i]
        align.alignto(u,ref,select='protein and backbone',weights='mass')
        W.write(sel)

models=cache.read_text()
view=py3Dmol.view(width=900,height=570)
view.setBackgroundColor('white')
view.addModelsAsFrames(models)

view.setStyle(
    {'protein':True},
    {'cartoon':{'color':'lightgray','opacity':float(cartoon_opacity)}}
)

view.addSurface(
    py3Dmol.VDW,
    {'opacity':float(surface_opacity),'color':'lightblue'},
    {'protein':True}
)

if show_ligand:
    view.setStyle(
        {'resn':'LIG'},
        {
            'stick':{'colorscheme':'orangeCarbon','radius':0.28},
            'sphere':{'colorscheme':'orangeCarbon','scale':0.14}
        }
    )

if show_waters:
    for rn in ['WAT','HOH','SOL']:
        view.setStyle(
            {'resn':rn},
            {'stick':{'radius':0.08,'opacity':0.38}}
        )
        view.setStyle(
            {'resn':rn,'elem':'O'},
            {'sphere':{'radius':0.16,'opacity':0.40}}
        )

view.zoomTo({'resn':'LIG'})
view.animate({'loop':'forward','interval':55})
view.show()

print('Frames mostrados:',len(frame_idx))
print('Moléculas de agua seleccionadas:',len(selected_water_resindices))
print('Estas aguas se eligieron por proximidad al ligando en el primer frame y son solo una muestra visual.')


In [ ]:
#@title Explorar un frame y el solvente cercano
frame_fraction = 0.50 #@param {type:"slider", min:0.0, max:1.0, step:0.05}
water_cutoff_A = 8.0 #@param {type:"slider", min:4.0, max:14.0, step:0.5}
protein_surface_opacity = 0.14 #@param {type:"slider", min:0.05, max:0.35, step:0.01}

import MDAnalysis as mda, numpy as np, py3Dmol
from MDAnalysis.lib.distances import distance_array

D=REF_LOADED/'systems'/student_slug
R=REF_LOADED/'runs'/student_slug/'production'
trjs=[str(p) for p in sorted(R.glob('traj_*.xtc'))]
u=mda.Universe(str(D/'system.prmtop'),trjs)

idx=int(round(float(frame_fraction)*(len(u.trajectory)-1)))
u.trajectory[idx]

lig=u.select_atoms('resname LIG')
prot=u.select_atoms('protein')
watO=u.select_atoms(
    '(resname WAT or resname HOH or resname SOL) and '
    '(name O or name OW or name OH2)'
)

selected_resindices=[]
if len(watO) and len(lig):
    d=distance_array(watO.positions,lig.positions,box=u.dimensions)
    close=np.where(np.any(d <= float(water_cutoff_A),axis=1))[0]
    selected_resindices=sorted(set(int(x) for x in watO[close].resindices))

wat=(
    u.atoms[np.isin(u.atoms.resindices,selected_resindices)]
    if selected_resindices else u.atoms[[]]
)

sel=prot+lig+wat
pdbtmp=ROOT/'selected_frame_with_water.pdb'
with mda.Writer(str(pdbtmp),sel.n_atoms) as W:
    W.write(sel)

view=py3Dmol.view(width=900,height=560)
view.setBackgroundColor('white')
view.addModel(pdbtmp.read_text(),'pdb')
view.setStyle({'protein':True},{'cartoon':{'color':'lightgray','opacity':0.92}})
view.addSurface(
    py3Dmol.VDW,
    {'opacity':float(protein_surface_opacity),'color':'lightblue'},
    {'protein':True}
)
view.setStyle(
    {'resn':'LIG'},
    {'stick':{'colorscheme':'orangeCarbon','radius':0.28},
     'sphere':{'colorscheme':'orangeCarbon','scale':0.14}}
)
for rn in ['WAT','HOH','SOL']:
    view.setStyle({'resn':rn},{'stick':{'radius':0.09,'opacity':0.48}})
    view.setStyle({'resn':rn,'elem':'O'},{'sphere':{'radius':0.17,'opacity':0.45}})

view.zoomTo({'resn':'LIG'})
view.show()

print('Frame:',idx,'de',len(u.trajectory)-1)
print('Aguas dentro de',water_cutoff_A,'A del ligando:',len(selected_resindices))
if len(selected_resindices)==0:
    print('La cavidad puede permanecer seca con este cutoff. Prueba un cutoff mayor y vuelve a ejecutar.')


## Qué nos dice la trayectoria

Una trayectoria es una secuencia de coordenadas:

$$
\mathbf R(t_0),\mathbf R(t_1),\mathbf R(t_2),\ldots
$$

El análisis transforma esas coordenadas en preguntas cuantitativas. Ninguna métrica por sí sola decide cuál ligando es "mejor".

### RMSD

Después de alinear las estructuras, el RMSD mide cuánto se separa un conjunto de átomos de una referencia:

$$
\mathrm{RMSD}(t)
=
\sqrt{
\frac{1}{N}
\sum_{i=1}^{N}
\left\|
\mathbf r_i(t)-\mathbf r_i^{\,ref}
\right\|^2
}
$$

En este notebook calculamos RMSD de C-alpha para la proteína y RMSD del ligando después de alinear la proteína.

Un RMSD bajo no significa automáticamente mayor afinidad.

### Distancia ligando-pocket

Podemos seguir la distancia entre el centro del ligando y el centro del pocket:

$$
d(t)
=
\left\|
\mathbf R_{\mathrm{lig}}(t)
-
\mathbf R_{\mathrm{pocket}}(t)
\right\|
$$

Esto responde una pregunta distinta al RMSD. Un ligando puede rotar dentro de la cavidad, aumentar su RMSD y aun así permanecer cerca del pocket.

### RMSF

Para cada residuo podemos preguntar cuánto fluctúa alrededor de su posición promedio:

$$
\mathrm{RMSF}_i
=
\sqrt{
\left\langle
\left\|
\mathbf r_i(t)-\langle\mathbf r_i\rangle
\right\|^2
\right\rangle
}
$$

RMSD compara frames con una referencia. RMSF identifica regiones particularmente móviles.

### Contact occupancy

Para un residuo definimos si existe un contacto en cada frame. La ocupación es:

$$
\mathrm{occupancy}
=
\frac{N_{\mathrm{frames\ con\ contacto}}}
{N_{\mathrm{frames\ totales}}}
$$

Esto convierte la pregunta estática del Día 2, "qué residuo toca al ligando", en una pregunta dinámica: "qué contacto aparece repetidamente durante la trayectoria".

### Pocket hydration

También podemos contar cuántas moléculas de agua visitan una región alrededor del ligando:

$$
N_{\mathrm{water}}(t;r_c)
=
\sum_j
I\left(d_j(t)\le r_c\right)
$$

donde $r_c$ es el cutoff que elegimos e $I$ vale 1 cuando una molécula de agua está dentro de esa distancia.

Ahora sí podemos usar las gráficas para volver a la pregunta biológica.


In [ ]:
#@title Explorar una métrica de la trayectoria
analysis_choice = "Protein RMSD" #@param ["Protein RMSD", "Ligand RMSD", "Ligand-pocket distance", "Pocket RMSD", "Pocket hydration at 5 A"]
smooth_window = 5 #@param {type:"slider", min:1, max:51, step:2}

import pandas as pd, matplotlib.pyplot as plt
A=REF_LOADED/'analysis'/student_slug
df=pd.read_csv(A/'timeseries.csv')
cols={
 'Protein RMSD':('protein_CA_RMSD_A','Protein C-alpha RMSD (A)'),
 'Ligand RMSD':('ligand_RMSD_A','Ligand RMSD (A)'),
 'Ligand-pocket distance':('ligand_pocket_distance_A','Ligand-pocket distance (A)'),
 'Pocket RMSD':('pocket_CA_RMSD_A','Pocket C-alpha RMSD (A)'),
 'Pocket hydration at 5 A':('waters_near_ligand','Waters within 5 A of ligand')}
col,y=cols[analysis_choice]
x=df.time_ps/1000.0
yv=df[col].rolling(int(smooth_window),center=True,min_periods=1).mean()
plt.figure(figsize=(9,3.8)); plt.plot(x,yv); plt.xlabel('Time (ns)'); plt.ylabel(y); plt.title(selected_ligand+' | '+analysis_choice); plt.show()
print('Mean:',round(float(df[col].mean()),3),'| Final:',round(float(df[col].iloc[-1]),3))

if analysis_choice == 'Pocket hydration at 5 A' and float(df[col].mean()) == 0:
    print('Interpretación: con un cutoff estricto de 5 A, la cavidad muestreada puede estar seca. Esto no significa que el sistema no tenga solvente.')


In [ ]:
#@title Ver contactos persistentes
minimum_occupancy = 0.30 #@param {type:"slider", min:0.05, max:1.0, step:0.05}
import pandas as pd, matplotlib.pyplot as plt
A=REF_LOADED/'analysis'/student_slug
c=pd.read_csv(A/'contact_occupancy.csv')
c=c[c.occupancy>=float(minimum_occupancy)].head(20)
display(c)
if len(c):
    plt.figure(figsize=(8,max(3,0.3*len(c))))
    plt.barh(c.residue[::-1],c.occupancy[::-1]); plt.xlim(0,1); plt.xlabel('Fraction of frames'); plt.ylabel('Residue'); plt.show()
else:
    print('No hay contactos por encima de ese cutoff.')


In [ ]:
#@title Explorar flexibilidad por residuo con RMSF
highlight_pocket = True #@param {type:"boolean"}

import json, pandas as pd, matplotlib.pyplot as plt
A=REF_LOADED/'analysis'/student_slug

rmsf_df=pd.read_csv(A/'protein_CA_RMSF.csv')
meta=json.loads((A/'analysis_metadata.json').read_text())
pocket_resids=set(int(x) for x in meta.get('pocket_resids',[]))

plt.figure(figsize=(10,3.8))
plt.plot(rmsf_df.resid,rmsf_df.RMSF_A,label='C-alpha RMSF')

if highlight_pocket and pocket_resids:
    mask=rmsf_df.resid.isin(pocket_resids)
    plt.scatter(
        rmsf_df.loc[mask,'resid'],
        rmsf_df.loc[mask,'RMSF_A'],
        s=28,
        label='Pocket'
    )

plt.xlabel('Residue')
plt.ylabel('RMSF (A)')
plt.title(selected_ligand+' | flexibilidad por residuo')
plt.legend()
plt.show()

print('RMSF alto significa mayor fluctuación alrededor de la posición promedio durante esta trayectoria.')
print('No implica por sí solo que un residuo sea funcionalmente más importante.')


In [ ]:
#@title Comparar los tres ligandos
import pandas as pd, matplotlib.pyplot as plt
rows=[]
for name in ['benzene','toluene','n_propylbenzene']:
    d=pd.read_csv(REF_LOADED/'analysis'/name/'timeseries.csv')
    rows.append({'Ligand':name,'Protein RMSD':d.protein_CA_RMSD_A.mean(),'Ligand RMSD':d.ligand_RMSD_A.mean(),
                 'Pocket distance':d.ligand_pocket_distance_A.mean(),'Pocket RMSD':d.pocket_CA_RMSD_A.mean(),
                 'Waters within 5 A':d.waters_near_ligand.mean()})
comparison=pd.DataFrame(rows)
display(comparison.round(3))


In [ ]:
#@title Hipótesis final antes de ver las estructuras experimentales
final_1 = "Toluene" #@param ["Benzene", "Toluene", "n-Propylbenzene"]
final_2 = "Benzene" #@param ["Benzene", "Toluene", "n-Propylbenzene"]
final_3 = "n-Propylbenzene" #@param ["Benzene", "Toluene", "n-Propylbenzene"]
confidence_day3 = 3 #@param {type:"slider", min:1, max:5, step:1}
most_useful_evidence = "Ligand behavior in MD" #@param ["Trajectory visualization", "Protein RMSD", "Ligand behavior in MD", "Pocket distance", "Persistent contacts", "Pocket hydration", "Combination of evidence"]

ranking=[final_1,final_2,final_3]
if len(set(ranking))<3:
    print('Usa cada ligando una sola vez.')
else:
    print('Ranking final del grupo:')
    for i,x in enumerate(ranking,1): print(i,x)
    print('Evidencia con más peso:',most_useful_evidence)
    print('Confianza:',confidence_day3,'/5')


# Parte III | Comparar la predicción con el experimento

Hasta este punto no usamos las estructuras experimentales de los complejos para decidir la respuesta.

Antes de revelar una estructura, su grupo debe haber registrado:

1. ranking final;
2. confianza de 1 a 5;
3. evidencia que más influyó;
4. una observación concreta de la trayectoria.

Ahora sí vamos a comparar la predicción con cristalografía de rayos X.

Los tres complejos experimentales que usaremos son:

- Benzene: **4W52**
- Toluene: **4W53**
- n-Propylbenzene: **4W55**

Los archivos cristalográficos también contienen otras moléculas de la condición experimental, por ejemplo HEPES y agua. En la visualización siguiente ocultamos esos heterógenos y mostramos **solo el ligando que corresponde a nuestro experimento**.


In [ ]:
#@title Revelar una estructura experimental
import requests, py3Dmol

experimental={
    'Benzene':('4W52','BNZ'),
    'Toluene':('4W53','MBN'),
    'n-Propylbenzene':('4W55','3H0')
}

experimental_ligand = "Toluene" #@param ["Benzene", "Toluene", "n-Propylbenzene"]
experimental_surface_opacity = 0.14 #@param {type:"slider", min:0.05, max:0.35, step:0.01}

pdbid,lig_resname=experimental[experimental_ligand]
r=requests.get(f'https://files.rcsb.org/download/{pdbid}.pdb',timeout=60)
r.raise_for_status()
pdbtxt=r.text

view=py3Dmol.view(width=900,height=560)
view.setBackgroundColor('white')
view.addModel(pdbtxt,'pdb')

# Nothing is drawn by default. We explicitly show protein and the intended ligand.
view.setStyle(
    {'protein':True},
    {'cartoon':{'color':'lightgray','opacity':0.92}}
)
view.addSurface(
    py3Dmol.VDW,
    {'opacity':float(experimental_surface_opacity),'color':'lightblue'},
    {'protein':True}
)
view.setStyle(
    {'resn':lig_resname},
    {
        'stick':{'colorscheme':'greenCarbon','radius':0.30},
        'sphere':{'colorscheme':'greenCarbon','scale':0.15}
    }
)

view.zoomTo({'resn':lig_resname})
view.show()

print('Ligando:',experimental_ligand)
print('PDB:',pdbid)
print('Residue name del ligando:',lig_resname)
print('Otros heterógenos del cristal están ocultos intencionalmente.')


### Volver a la hipótesis

Comparen mentalmente tres cosas:

$$
\text{pose de GNINA}
\quad\rightarrow\quad
\text{trayectoria MD}
\quad\rightarrow\quad
\text{estructura experimental}
$$

Discusión breve:

- qué cambió entre la pose inicial y la trayectoria;
- qué métrica ayudó más a interpretar ese cambio;
- qué parte del razonamiento coincidió con el experimento;
- qué conclusión no sería válida hacer a partir de una trayectoria corta.

Una predicción computacional sigue siendo una hipótesis evaluable. El experimento es otra capa de evidencia.


# Capstone | Qué puede revelar una trayectoria real?

Antes de elegir su propio target, vamos a aplicar lo aprendido a un segundo sistema biomédico.

**HIV-1 protease** es una proteasa dimérica necesaria para la maduración del virus. Dos regiones flexibles llamadas **flaps** cubren el sitio activo.

Los extremos de las dos flaps contienen **Ile50** e **Ile50'**.

Una forma simple de seguir la apertura es medir:

$$
d_{\mathrm{flap}}(t)
=
\left\|
\mathbf r_{\mathrm{Ile50}}(t)
-
\mathbf r_{\mathrm{Ile50'}}(t)
\right\|
$$

Ahora la pregunta es:

> Puede una sola distancia convertir una película molecular complicada en una señal que podamos interpretar?

Usaremos una trayectoria pública de HIV-1 protease asociada a un estudio de cambios conformacionales. El paquete completo que descargamos pesa aproximadamente 85 MB y contiene trayectorias XTC reales y estructuras iniciales.

Fuente: Li H, Ma A. *Enhanced Sampling of Protein Conformational Changes via True Reaction Coordinates from Energy Relaxation.* Datos públicos: https://doi.org/10.5281/zenodo.14531159


In [ ]:
#@title Cargar una trayectoria publicada de HIV-1 protease
import urllib.request, zipfile, shutil, os, re
from pathlib import Path
import MDAnalysis as mda

HIV_DIR = ROOT / 'hiv_capstone'
HIV_ZIP = ROOT / 'hiv_traj_data.zip'
HIV_URL = 'https://zenodo.org/records/14531159/files/traj_data.zip?download=1'

if not HIV_ZIP.exists():
    print('Descargando dataset público de HIV-1 protease (~85 MB)...')
    urllib.request.urlretrieve(HIV_URL, HIV_ZIP)

if not HIV_DIR.exists():
    HIV_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(HIV_ZIP) as z:
        z.extractall(HIV_DIR)

xtc_candidates = sorted(
    p for p in HIV_DIR.rglob('*.xtc')
    if 'DRV' in p.name.upper() or 'DRV' in str(p.parent).upper()
)

# Prefer a natural reactive trajectory if the archive contains one.
nrt = [p for p in xtc_candidates if 'NRT' in p.name.upper() or 'NRT' in str(p.parent).upper()]
if nrt:
    xtc_candidates = nrt + [p for p in xtc_candidates if p not in nrt]

pdb_candidates = sorted(
    p for p in HIV_DIR.rglob('*.pdb')
    if 'DRV' in p.name.upper() or 'DRV' in str(p.parent).upper()
)

initials = [p for p in pdb_candidates if 'INITIAL' in p.name.upper()]
if initials:
    pdb_candidates = initials + [p for p in pdb_candidates if p not in initials]

if not xtc_candidates:
    raise FileNotFoundError('No encontré una trayectoria DRV en el ZIP.')

# Find a topology/trajectory pair with matching atom counts.
HIV_UNIVERSE = None
HIV_TOPOLOGY = None
HIV_TRAJECTORY = None

for xtc in xtc_candidates:
    for pdb in pdb_candidates:
        try:
            test_u = mda.Universe(str(pdb), str(xtc))
            # Require a protease-like protein and at least one frame.
            if len(test_u.select_atoms('protein')) > 100 and len(test_u.trajectory) > 1:
                HIV_UNIVERSE = test_u
                HIV_TOPOLOGY = pdb
                HIV_TRAJECTORY = xtc
                break
        except Exception:
            continue
    if HIV_UNIVERSE is not None:
        break

if HIV_UNIVERSE is None:
    raise RuntimeError(
        'El dataset se descargó, pero no pude emparejar automáticamente el PDB y XTC de DRV.'
    )

print('Topología:', HIV_TOPOLOGY.name)
print('Trayectoria:', HIV_TRAJECTORY.name)
print('Frames:', len(HIV_UNIVERSE.trajectory))
print('Átomos:', HIV_UNIVERSE.atoms.n_atoms)


In [ ]:
#@title Medir la apertura de las flaps de HIV-1 protease
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import MDAnalysis as mda
from MDAnalysis.analysis import align

u = mda.Universe(str(HIV_TOPOLOGY), str(HIV_TRAJECTORY))
ref = mda.Universe(str(HIV_TOPOLOGY))
ref.trajectory[0]

# Two Ile50 C-alpha atoms, one from each protease monomer.
flap_atoms = u.select_atoms('protein and name CA and resid 50')

if len(flap_atoms) != 2:
    # Fallback: collect one resid 50 CA from each segment/chain.
    found = []
    for seg in u.segments:
        ag = seg.atoms.select_atoms('protein and name CA and resid 50')
        if len(ag):
            found.append(ag[0].index)
    if len(found) >= 2:
        flap_atoms = u.atoms[found[:2]]

if len(flap_atoms) != 2:
    raise RuntimeError(
        f'Esperaba dos C-alpha de Ile50 y encontré {len(flap_atoms)}. '
        'Revisa la numeración del dataset.'
    )

rows = []
for i, ts in enumerate(u.trajectory):
    # Alignment makes the visual easier to interpret, but does not change the
    # distance between the two Ile50 atoms.
    try:
        align.alignto(u, ref, select='protein and backbone', weights='mass')
    except Exception:
        pass

    d = np.linalg.norm(flap_atoms.positions[0] - flap_atoms.positions[1])
    rows.append({'frame': i, 'time_ps': float(ts.time), 'flap_distance_A': float(d)})

hiv_flap_df = pd.DataFrame(rows)

i_min = int(hiv_flap_df.flap_distance_A.idxmin())
i_max = int(hiv_flap_df.flap_distance_A.idxmax())

plt.figure(figsize=(10, 4))
plt.plot(hiv_flap_df.frame, hiv_flap_df.flap_distance_A)
plt.scatter(
    [hiv_flap_df.loc[i_min, 'frame'], hiv_flap_df.loc[i_max, 'frame']],
    [hiv_flap_df.loc[i_min, 'flap_distance_A'], hiv_flap_df.loc[i_max, 'flap_distance_A']],
    s=55
)
plt.xlabel('Frame')
plt.ylabel("Ile50 - Ile50' distance (A)")
plt.title('HIV-1 protease | distancia entre las puntas de las flaps')
plt.show()

print('Distancia mínima:', round(float(hiv_flap_df.flap_distance_A.min()), 2), 'A')
print('Distancia máxima:', round(float(hiv_flap_df.flap_distance_A.max()), 2), 'A')
print('Cambio observado:', round(float(hiv_flap_df.flap_distance_A.max()-hiv_flap_df.flap_distance_A.min()), 2), 'A')

HIV_FRAME_MIN = int(hiv_flap_df.loc[i_min, 'frame'])
HIV_FRAME_MAX = int(hiv_flap_df.loc[i_max, 'frame'])


In [ ]:
#@title Ver la trayectoria y localizar el evento en 3D
hiv_view = "Animación" #@param ["Animación", "Frame con menor distancia", "Frame con mayor apertura"]
hiv_surface_opacity = 0.12 #@param {type:"slider", min:0.03, max:0.30, step:0.01}

import MDAnalysis as mda, numpy as np, py3Dmol
from MDAnalysis.analysis import align
from pathlib import Path

u = mda.Universe(str(HIV_TOPOLOGY), str(HIV_TRAJECTORY))
ref = mda.Universe(str(HIV_TOPOLOGY))
ref.trajectory[0]

def ligand_resname(universe):
    standard = {
        'ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE',
        'LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL',
        'WAT','HOH','SOL','NA','CL','K','MG','CA'
    }
    candidates = []
    for res in universe.residues:
        if res.resname.upper() not in standard and len(res.atoms) >= 10:
            candidates.append((len(res.atoms), res.resname))
    return sorted(candidates, reverse=True)[0][1] if candidates else None

lig_resn = ligand_resname(u)

if hiv_view == 'Animación':
    idx = np.unique(np.linspace(0, len(u.trajectory)-1, min(100, len(u.trajectory))).astype(int))
    temp = ROOT / 'hiv_viewer.pdb'
    selection = u.select_atoms('protein')
    if lig_resn:
        selection = selection + u.select_atoms(f'resname {lig_resn}')

    with mda.Writer(str(temp), selection.n_atoms, multiframe=True) as W:
        for i in idx:
            u.trajectory[i]
            try:
                align.alignto(u, ref, select='protein and backbone', weights='mass')
            except Exception:
                pass
            W.write(selection)

    view = py3Dmol.view(width=900, height=560)
    view.setBackgroundColor('white')
    view.addModelsAsFrames(temp.read_text())
    view.setStyle({'protein':True},{'cartoon':{'color':'lightgray','opacity':0.92}})
    view.addSurface(py3Dmol.VDW,{'opacity':float(hiv_surface_opacity),'color':'lightblue'},{'protein':True})
    view.setStyle({'resi':50},{'stick':{'colorscheme':'yellowCarbon','radius':0.24}})
    if lig_resn:
        view.setStyle({'resn':lig_resn},{'stick':{'colorscheme':'orangeCarbon','radius':0.25}})
    view.zoomTo({'resi':50})
    view.animate({'loop':'forward','interval':65})
    view.show()

else:
    frame = HIV_FRAME_MIN if hiv_view == 'Frame con menor distancia' else HIV_FRAME_MAX
    u.trajectory[frame]
    try:
        align.alignto(u, ref, select='protein and backbone', weights='mass')
    except Exception:
        pass

    selection = u.select_atoms('protein')
    if lig_resn:
        selection = selection + u.select_atoms(f'resname {lig_resn}')

    temp = ROOT / 'hiv_selected_frame.pdb'
    with mda.Writer(str(temp), selection.n_atoms) as W:
        W.write(selection)

    view = py3Dmol.view(width=900, height=560)
    view.setBackgroundColor('white')
    view.addModel(temp.read_text(),'pdb')
    view.setStyle({'protein':True},{'cartoon':{'color':'lightgray','opacity':0.92}})
    view.addSurface(py3Dmol.VDW,{'opacity':float(hiv_surface_opacity),'color':'lightblue'},{'protein':True})
    view.setStyle({'resi':50},{'stick':{'colorscheme':'yellowCarbon','radius':0.28}})
    if lig_resn:
        view.setStyle({'resn':lig_resn},{'stick':{'colorscheme':'orangeCarbon','radius':0.28}})
    view.zoomTo({'resi':50})
    view.show()

    value = float(hiv_flap_df.loc[hiv_flap_df.frame == frame,'flap_distance_A'].iloc[0])
    print('Frame:',frame)
    print("Ile50-Ile50' distance:",round(value,2),'A')


### Qué reveló el análisis?

Al principio teníamos una película con muchos átomos moviéndose.

Después elegimos una pregunta biológica concreta y la convertimos en una coordenada medible:

$$
\text{movimiento de las flaps}
\rightarrow
d_{\mathrm{Ile50-Ile50'}}
$$

La gráfica nos permitió localizar frames con diferente separación y volver a la estructura 3D para interpretarlos.

Ese es uno de los patrones más útiles en análisis de MD:

$$
\boxed{
\text{mirar}
\rightarrow
\text{formular una pregunta}
\rightarrow
\text{medir}
\rightarrow
\text{volver a mirar}
}
$$

En estudios clásicos de HIV-1 protease, la dinámica de las flaps se ha relacionado con el acceso al sitio activo y con estados abiertos, semiabiertos y cerrados. Una trayectoria particular no representa por sí sola todo el ensemble de la proteína.


# Parte IV | Tu propia pregunta biomédica

Para cerrar, cada grupo elegirá un target humano relacionado con una enfermedad y preparará una MD corta de su estructura predicha.

## Paso 1 | Open Targets

Abre [Open Targets Platform](https://platform.opentargets.org/).

1. Busca una enfermedad que te interese.
2. Entra a la página de la enfermedad.
3. Revisa los **associated targets**.
4. Elige un target y revisa qué tipos de evidencia contribuyen a su asociación con la enfermedad.
5. Anota el símbolo del gen y el nombre de la proteína.

Open Targets integra múltiples fuentes para priorizar asociaciones target-enfermedad. Un score alto no demuestra por sí solo causalidad ni que el target sea fácil de convertir en fármaco.

## Paso 2 | Elegir un target que podamos simular hoy

Para que esta actividad funcione en Colab, intenta escoger:

- proteína humana;
- una sola cadena;
- aproximadamente 100 a 500 aminoácidos;
- soluble;
- sin un segmento transmembrana evidente;
- sin depender obligatoriamente de un metal, cofactor o complejo grande para mantener su estructura.

Si tu primera opción es un receptor de membrana o una proteína enorme, conserva ese target como hallazgo biológico pero elige otro target asociado para la simulación de hoy.

## Paso 3 | UniProt y AlphaFold

1. Busca el target en [UniProt](https://www.uniprot.org/).
2. Confirma que el organismo sea **Homo sapiens**.
3. Anota el **UniProt accession**.
4. En la sección **Structure**, abre el enlace de **AlphaFold DB**.
5. En AlphaFold DB descarga la estructura en formato **PDB**.

AlphaFold asigna a cada residuo un valor de confianza llamado pLDDT:

$$
0 \leq \mathrm{pLDDT} \leq 100
$$

Como guía:

$$
\begin{aligned}
\mathrm{pLDDT} &> 90 && \text{confianza muy alta}\\
70 < \mathrm{pLDDT} &\leq 90 && \text{confianza alta}\\
50 < \mathrm{pLDDT} &\leq 70 && \text{confianza baja}\\
\mathrm{pLDDT} &\leq 50 && \text{confianza muy baja}
\end{aligned}
$$

En los PDB descargados de AlphaFold DB, pLDDT se guarda en el campo que tradicionalmente se usa para B-factor.

Primero vamos a inspeccionar la confianza. Después decidimos si tiene sentido simular ese modelo.


In [ ]:
#@title Registrar la elección del grupo
disease_name = "" #@param {type:"string"}
target_gene = "" #@param {type:"string"}
target_protein_name = "" #@param {type:"string"}
uniprot_accession = "" #@param {type:"string"}
target_reason = "" #@param {type:"string"}

print('Enfermedad:',disease_name or '[completa este campo]')
print('Target:',target_gene or '[completa este campo]')
print('Proteína:',target_protein_name or '[completa este campo]')
print('UniProt:',uniprot_accession or '[completa este campo]')
print('Razón de selección:',target_reason or '[completa este campo]')


In [ ]:
#@title Subir el PDB o CIF de AlphaFold
from google.colab import files
from pathlib import Path
import shutil

print("Selecciona tu archivo .pdb o .cif si deseas subir uno nuevo. Si cancelas, intentaremos usar uno existente en /content.")
try:
    uploaded = files.upload()
except Exception as e:
    print("No se completó la subida interactiva:", e)
    uploaded = {}

# Accept both .pdb and .cif extensions
valid_names = [name for name in uploaded if name.lower().endswith(('.pdb', '.cif'))]

if not valid_names:
    # Fallback to look for files in /content if upload was empty/cancelled
    local_files = list(Path('/content').glob('*.cif')) + list(Path('/content').glob('*.pdb'))
    if local_files:
        name = local_files[0].name
        print(f"Usando archivo existente encontrado en /content: {name}")
        AF_FILE = local_files[0]
    else:
        raise ValueError('Sube un archivo .pdb o .cif descargado de AlphaFold DB o colócalo en /content.')
else:
    name = valid_names[0]
    AF_FILE = Path('/content')/name
    AF_FILE.write_bytes(uploaded[name])

TARGET_ROOT = Path('/content/day3_independent_target')
TARGET_ROOT.mkdir(parents=True, exist_ok=True)

# If it is a CIF file, we will store it as input and make sure later steps understand it
if name.lower().endswith('.cif'):
    TARGET_INPUT = TARGET_ROOT/'alphafold_input.cif'
else:
    TARGET_INPUT = TARGET_ROOT/'alphafold_input.pdb'

shutil.copy2(AF_FILE, TARGET_INPUT)
print('Archivo cargado y listo en:', TARGET_INPUT)

In [ ]:
#@title Revisar confianza y estructura del modelo AlphaFold
import pandas as pd, numpy as np, py3Dmol
from pathlib import Path

if 'TARGET_INPUT' not in globals() or not Path(TARGET_INPUT).exists():
    raise RuntimeError('Primero sube el archivo PDB de AlphaFold.')

pdbtxt=Path(TARGET_INPUT).read_text()
rows=[]

for line in pdbtxt.splitlines():
    if not line.startswith('ATOM'):
        continue
    if line[12:16].strip()!='CA':
        continue
    try:
        rows.append({
            'chain':line[21].strip() or 'A',
            'resid':int(line[22:26]),
            'resname':line[17:20].strip(),
            'pLDDT':float(line[60:66])
        })
    except Exception:
        pass

plddt=pd.DataFrame(rows)
if plddt.empty:
    raise ValueError('No pude leer residuos C-alpha ni pLDDT del PDB.')

nres=len(plddt)
mean_plddt=float(plddt.pLDDT.mean())
very_high=(plddt.pLDDT>=90).mean()*100
high=((plddt.pLDDT>=70)&(plddt.pLDDT<90)).mean()*100
low=((plddt.pLDDT>=50)&(plddt.pLDDT<70)).mean()*100
very_low=(plddt.pLDDT<50).mean()*100

print('Residuos:',nres)
print('pLDDT promedio:',round(mean_plddt,1))
print('Muy alta >=90:',round(very_high,1),'%')
print('Alta 70-90:',round(high,1),'%')
print('Baja 50-70:',round(low,1),'%')
print('Muy baja <50:',round(very_low,1),'%')

if nres>500:
    print()
    print('Advertencia: esta proteína tiene más de 500 residuos y puede ser lenta para la actividad de hoy.')
if (plddt.pLDDT<70).mean()>0.30:
    print()
    print('Advertencia: más de 30% del modelo tiene pLDDT menor de 70. Interpreta esas regiones con cautela.')

groups={
    '#0053D6':plddt.loc[plddt.pLDDT>=90,'resid'].tolist(),
    '#65CBF3':plddt.loc[(plddt.pLDDT>=70)&(plddt.pLDDT<90),'resid'].tolist(),
    '#FFDB13':plddt.loc[(plddt.pLDDT>=50)&(plddt.pLDDT<70),'resid'].tolist(),
    '#FF7D45':plddt.loc[plddt.pLDDT<50,'resid'].tolist()
}

view=py3Dmol.view(width=900,height=560)
view.setBackgroundColor('white')
view.addModel(pdbtxt,'pdb')

for color,resids in groups.items():
    if resids:
        view.setStyle({'resi':resids},{'cartoon':{'color':color,'opacity':0.95}})

view.addSurface(py3Dmol.VDW,{'opacity':0.10,'color':'white'},{'protein':True})
view.zoomTo()
view.show()

print('Azul oscuro: >=90 | azul claro: 70-90 | amarillo: 50-70 | naranja: <50')


## Preparar la simulación del target

Para esta extensión usaremos una simulación de **proteína sola en agua**, sin ligando.

El workflow es:

$$
\text{AlphaFold PDB}
\rightarrow
\text{hidrógenos}
\rightarrow
\text{agua + iones}
\rightarrow
\text{minimización}
\rightarrow
\text{equilibración}
\rightarrow
\text{producción corta}
$$

Esto no convierte al modelo de AlphaFold en una estructura experimental y tampoco demuestra que el target sea terapéuticamente válido. La pregunta es más modesta:

> **Cómo se comporta este modelo estructural durante una MD corta bajo las condiciones que elegimos?**


In [ ]:
#@title Preparar el sistema de tu target
target_temperature_K = 310.0 #@param {type:"slider", min:285.0, max:325.0, step:1.0}
target_pressure_bar = 1.0 #@param {type:"slider", min:0.8, max:1.2, step:0.05}
target_salt_M = 0.15 #@param {type:"slider", min:0.0, max:0.30, step:0.01}
target_pH = 7.4 #@param {type:"slider", min:5.0, max:9.0, step:0.1}
target_padding_A = "10" #@param [8.0, 10.0, 12.0]

from pathlib import Path
import openmm as mm
from openmm import unit
from openmm.app import PDBFile, Modeller, ForceField, PME, HBonds

if 'TARGET_INPUT' not in globals():
    raise RuntimeError('Primero sube el PDB de AlphaFold.')

TARGET_ROOT=Path('/content/day3_independent_target')
TARGET_ROOT.mkdir(parents=True,exist_ok=True)

# AlphaFold DB models are protein predictions. Keep ATOM records and remove accidental heterogens.
clean_lines=[
    line for line in Path(TARGET_INPUT).read_text().splitlines()
    if line.startswith('ATOM')
]
clean_lines += ['TER','END']
TARGET_CLEAN=TARGET_ROOT/'protein_only.pdb'
TARGET_CLEAN.write_text('\n'.join(clean_lines)+'\n')

pdb=PDBFile(str(TARGET_CLEAN))
modeller=Modeller(pdb.topology,pdb.positions)

try:
    forcefield=ForceField('amber14-all.xml','amber14/tip3pfb.xml')
    water_model='tip3p'
    ff_label='Amber14 + TIP3P-FB'
except Exception:
    forcefield=ForceField('amber14-all.xml','amber14/tip3p.xml')
    water_model='tip3p'
    ff_label='Amber14 + TIP3P'

protein_residues=[r for r in modeller.topology.residues()]
if len(protein_residues)>650:
    raise RuntimeError(
        f'El modelo tiene {len(protein_residues)} residuos. '
        'Para el tiempo disponible, elige un target más pequeño, idealmente 100 a 500 residuos.'
    )

print('Agregando hidrógenos a pH',target_pH)
modeller.addHydrogens(forcefield,pH=float(target_pH))

print('Agregando agua e iones...')
modeller.addSolvent(
    forcefield,
    model=water_model,
    padding=float(target_padding_A)*unit.angstrom,
    ionicStrength=float(target_salt_M)*unit.molar,
    neutralize=True
)

TARGET_PREPARED=TARGET_ROOT/'prepared_system.pdb'
with open(TARGET_PREPARED,'w') as f:
    PDBFile.writeFile(modeller.topology,modeller.positions,f,keepIds=True)

TARGET_SYSTEM=forcefield.createSystem(
    modeller.topology,
    nonbondedMethod=PME,
    nonbondedCutoff=1.0*unit.nanometer,
    constraints=HBonds,
    rigidWater=True
)
TARGET_SYSTEM.addForce(
    mm.MonteCarloBarostat(
        float(target_pressure_bar)*unit.bar,
        float(target_temperature_K)*unit.kelvin,
        25
    )
)

TARGET_TOPOLOGY=modeller.topology
TARGET_POSITIONS=modeller.positions

n_atoms=TARGET_TOPOLOGY.getNumAtoms()
n_res=sum(1 for _ in TARGET_TOPOLOGY.residues())

print('Force field:',ff_label)
print('Residuos totales después de solvatar:',n_res)
print('Átomos totales:',n_atoms)
print('Sistema listo para minimizar.')


In [ ]:
#@title Correr una MD corta de tu target
run_target_md = True #@param {type:"boolean"}
target_equil_ps = "10" #@param [5.0, 10.0, 20.0]
target_live_ps = "20" #@param [10.0, 20.0, 50.0, 100.0]
target_save_ps = "0.5" #@param [0.5, 1.0, 2.0]

if run_target_md:
    import time
    import openmm as mm
    from openmm import unit
    from openmm.app import Simulation, XTCReporter, StateDataReporter, PDBFile

    if 'TARGET_SYSTEM' not in globals():
        raise RuntimeError('Primero prepara el sistema de tu target.')

    names=[mm.Platform.getPlatform(i).getName() for i in range(mm.Platform.getNumPlatforms())]
    if 'CUDA' in names:
        platform=mm.Platform.getPlatformByName('CUDA'); props={'Precision':'mixed'}
    elif 'OpenCL' in names:
        platform=mm.Platform.getPlatformByName('OpenCL'); props={'Precision':'mixed'}
    else:
        platform=mm.Platform.getPlatformByName('CPU'); props={}

    if platform.getName()=='CPU':
        print('No se detectó GPU. Puedes reducir target_live_ps o dejar esta extensión para después del workshop.')

    integrator=mm.LangevinMiddleIntegrator(
        float(target_temperature_K)*unit.kelvin,
        1.0/unit.picosecond,
        2.0*unit.femtoseconds
    )

    simulation=Simulation(TARGET_TOPOLOGY,TARGET_SYSTEM,integrator,platform,props)
    simulation.context.setPositions(TARGET_POSITIONS)

    print('Plataforma:',platform.getName(),props)
    print('Minimizando...')
    simulation.minimizeEnergy(maxIterations=3000)

    simulation.context.setVelocitiesToTemperature(float(target_temperature_K)*unit.kelvin,2026)

    eq_steps=int(float(target_equil_ps)*1000/2.0)
    print('Equilibrando',target_equil_ps,'ps...')
    simulation.step(eq_steps)

    TARGET_START=TARGET_ROOT/'target_start.pdb'
    state=simulation.context.getState(getPositions=True,enforcePeriodicBox=True)
    with open(TARGET_START,'w') as f:
        PDBFile.writeFile(TARGET_TOPOLOGY,state.getPositions(),f,keepIds=True)

    TARGET_XTC=TARGET_ROOT/'target_trajectory.xtc'
    TARGET_LOG=TARGET_ROOT/'target_log.csv'

    report_steps=max(1,int(float(target_save_ps)*1000/2.0))
    prod_steps=int(float(target_live_ps)*1000/2.0)

    simulation.reporters=[
        XTCReporter(str(TARGET_XTC),report_steps,enforcePeriodicBox=True),
        StateDataReporter(
            str(TARGET_LOG),
            report_steps,
            step=True,time=True,potentialEnergy=True,
            temperature=True,density=True,speed=True,
            totalSteps=prod_steps,separator=','
        )
    ]

    t0=time.time()
    simulation.step(prod_steps)
    elapsed=time.time()-t0
    nsday=(float(target_live_ps)/1000)/(elapsed/86400)

    print('Tiempo simulado:',target_live_ps,'ps')
    print('Rendimiento:',round(nsday,1),'ns/day')
    print('Trayectoria:',TARGET_XTC)
else:
    print('Simulación omitida.')


In [ ]:
#@title Visualizar la trayectoria de tu target
target_viewer_frames = 70 #@param [35, 50, 70, 100]
target_surface_opacity = 0.14 #@param {type:"slider", min:0.05, max:0.35, step:0.01}

import numpy as np, MDAnalysis as mda, py3Dmol
from pathlib import Path
from MDAnalysis.analysis import align

if 'TARGET_XTC' not in globals() or not Path(TARGET_XTC).exists():
    raise RuntimeError('Primero corre la MD corta de tu target.')

u=mda.Universe(str(TARGET_START),str(TARGET_XTC))
ref=mda.Universe(str(TARGET_START))
ref.trajectory[0]
protein=u.select_atoms('protein')

idx=np.unique(
    np.linspace(0,len(u.trajectory)-1,min(int(target_viewer_frames),len(u.trajectory))).astype(int)
)

viewer_file=TARGET_ROOT/'target_viewer.pdb'
with mda.Writer(str(viewer_file),protein.n_atoms,multiframe=True) as W:
    for i in idx:
        u.trajectory[i]
        align.alignto(u,ref,select='protein and backbone',weights='mass')
        W.write(protein)

view=py3Dmol.view(width=900,height=560)
view.setBackgroundColor('white')
view.addModelsAsFrames(viewer_file.read_text())
view.setStyle({'protein':True},{'cartoon':{'color':'lightgray','opacity':0.95}})
view.addSurface(
    py3Dmol.VDW,
    {'opacity':float(target_surface_opacity),'color':'lightblue'},
    {'protein':True}
)
view.zoomTo()
view.animate({'loop':'forward','interval':60})
view.show()

print('Frames mostrados:',len(idx))


In [ ]:
#@title Medir RMSD de tu target
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
import MDAnalysis as mda
from MDAnalysis.analysis import align

if 'TARGET_XTC' not in globals() or not Path(TARGET_XTC).exists():
    raise RuntimeError('Primero corre la MD corta de tu target.')

u=mda.Universe(str(TARGET_START),str(TARGET_XTC))
ref=mda.Universe(str(TARGET_START))
ref.trajectory[0]

ca=u.select_atoms('protein and name CA')
refca=ref.select_atoms('protein and name CA')

rows=[]
for ts in u.trajectory:
    align.alignto(u,ref,select='protein and backbone',weights='mass')
    value=np.sqrt(np.mean(np.sum((ca.positions-refca.positions)**2,axis=1)))
    rows.append({'time_ps':float(ts.time),'CA_RMSD_A':float(value)})

target_rmsd_df=pd.DataFrame(rows)

plt.figure(figsize=(9,3.8))
plt.plot(target_rmsd_df.time_ps,target_rmsd_df.CA_RMSD_A)
plt.xlabel('Time (ps)')
plt.ylabel('C-alpha RMSD (A)')
plt.title((target_gene or target_protein_name or 'Target')+' | short MD')
plt.show()

print('RMSD promedio:',round(float(target_rmsd_df.CA_RMSD_A.mean()),3),'A')
print('Esta trayectoria es demasiado corta para afirmar convergencia. Úsala para describir el comportamiento observado.')


## Cierre del proyecto independiente

Guarden cuatro cosas:

1. enfermedad;
2. target y UniProt accession;
3. una observación de la estructura o trayectoria;
4. una limitación importante.

Una ruta posible para continuar este proyecto sería:

$$
\text{disease}
\rightarrow
\text{target evidence}
\rightarrow
\text{structure}
\rightarrow
\text{pocket}
\rightarrow
\text{ligand hypothesis}
\rightarrow
\text{docking / co-folding}
\rightarrow
\text{MD}
\rightarrow
\text{experiment}
$$

La MD no decide si un target causa una enfermedad ni si una molécula será un fármaco. Sirve para formular y evaluar preguntas estructurales dentro de un modelo físico y una escala de tiempo determinada.


# Referencias y recursos

- OpenMM: simulación molecular con soporte de CPU y GPU.
- Amber ff14SB y GAFF2: parámetros para proteína y ligandos usados en las referencias de T4.
- MDAnalysis: lectura, alineamiento y análisis de trayectorias.
- py3Dmol: visualización molecular interactiva.
- GNINA / Vinardo y P2Rank: protocolo de docking del Día 2.
- RCSB PDB: estructuras 4W51, 4W52, 4W53 y 4W55.
- Open Targets Platform: exploración de asociaciones entre targets y enfermedades.
- UniProt: secuencia, función, anotaciones y enlaces estructurales.
- AlphaFold Protein Structure Database: modelos estructurales predichos y pLDDT.

Recursos:

- https://platform.opentargets.org/
- https://www.uniprot.org/
- https://alphafold.ebi.ac.uk/
- https://www.rcsb.org/

AlphaFold DB permite descargar coordenadas en PDB o mmCIF. Su pLDDT es una medida de confianza local de 0 a 100 y se almacena en el campo B-factor de los archivos descargados.

Este notebook toma inspiración técnica de workflows abiertos de dinámica molecular en Google Colab, incluido **Making-it-rain** de Pablo R. Arantes y colaboradores, adaptado aquí para un workshop introductorio.

- KRAS G12C + AMG 510 MD dataset: Pantsar T. Scientific Reports 2020. https://doi.org/10.5281/zenodo.3711537
- HIV-1 protease reactive trajectory dataset: Li H, Ma A. https://doi.org/10.5281/zenodo.14531159
- HIV-1 protease flap dynamics background: Hornak V, Okur A, Rizzo RC, Simmerling C. PNAS 2006.

